# Solution B1 · Titanic: a complete pipeline, three variants compared

**Type**: Implement with scikit-learn

## Objective

Build with `ColumnTransformer` and `Pipeline` a model on Titanic that handles imputation, scaling and encoding by itself, evaluate it
in cross-validation, compare three preprocessing variants and read the coefficients with their names.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = ds.titanic()
X, y = df.drop(columns="survived"), df["survived"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
NUM = ["age", "siblings_spouses", "parents_children", "fare"]
CAT = ["sex", "port", "deck", "title"]
cv = StratifiedKFold(5, shuffle=True, random_state=0)

## Commented solution

In [ ]:
def build(num=NUM, cat=CAT, age_indicator=False):
    num_pipe = make_pipeline(SimpleImputer(strategy="median", add_indicator=age_indicator), StandardScaler())
    cat_pipe = make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore"))
    prep = ColumnTransformer([("num", num_pipe, num), ("cat", cat_pipe, cat), ("pclass", "passthrough", ["pclass"])],
                             sparse_threshold=0)
    return make_pipeline(prep, LogisticRegression(max_iter=2000))

In [ ]:
model = build()
cv_auc = cross_val_score(model, X_tr, y_tr, cv=cv, scoring="roc_auc").mean()
model.fit(X_tr, y_tr)
test_auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
coef = pd.Series(model[-1].coef_[0], index=model[:-1].get_feature_names_out())
top5 = coef.abs().nlargest(5).index.tolist()
print(f"cross-validated AUC: {cv_auc:.3f} | test AUC: {test_auc:.3f}")
print(coef[top5].round(2).to_string())

In [ ]:
def auc(m):
    return cross_val_score(m, X_tr, y_tr, cv=cv, scoring="roc_auc").mean()


table = pd.DataFrame({"cv_auc": {
    "base": auc(build()),
    "without_deck": auc(build(cat=[c for c in CAT if c != "deck"])),
    "with_age_indicator": auc(build(age_indicator=True)),
}})
table.round(4)

## Check

In [ ]:
with verify():
    check(0.80 < cv_auc < 0.90, "reasonable cross-validated AUC (between 0.80 and 0.90)", "with sex, title and pclass the AUC should exceed 0.8")
    check(0.80 < test_auc < 0.93, "reasonable AUC on the test set", "check that you trained on the training set and evaluated on the test set")
    check(len(top5) == 5 and any("sex" in n or "title" in n for n in top5), "sex or title appear among the five most influential features",
          "top5 = names sorted by |coefficient|, with the names from get_feature_names_out")
    check(set(table.index) == {"base", "without_deck", "with_age_indicator"} and table["cv_auc"].notna().all(), "table with the three variants",
          "rows: base, without_deck, with_age_indicator (in any order)")
    check(close(table.loc["base", "cv_auc"], cv_auc), "the base variant matches cv_auc", "the base variant is the model of step 2")
    check(abs(table.loc["without_deck", "cv_auc"] - table.loc["base", "cv_auc"]) < 0.02,
          "removing the deck (77% missing) changes the AUC little", "the deck is almost all missing: removing it changes the AUC by less than 0.02")
done()

## What you learned

- The scheme is always the same: `ColumnTransformer` per group of columns, every group with its own mini-pipeline (imputation and then
  transformation), a final `Pipeline` with the model, cross-validation on the training set and the test set touched **once**.
- The variants are compared with the same `cv` and the same function: here the differences (deck, `age` indicator) are minimal, so the
  choice of complexity is not justified by the data. A clean comparison also serves to **not** add useless complications.
- The names with prefixes (`num__age`, `cat__sex_female`) let you read the model: sex and title dominate, as in the
  story of the Titanic.

## Common pitfall

Comparing variants by looking at the test set at every trial. The test set is used once; to choose among variants you use
cross-validation on the training set.

## Interview question

<details><summary>Why do imputation and scaling sit inside a Pipeline and not in a separate step before the split?</summary>

Because they are steps that learn from the data (median, mean, standard deviation): if I run them before the split, the test set
contributes to the statistics. Inside a Pipeline every cross-validation fold refits them on the training part of the fold alone, and in production
the same object applies exactly the statistics learned during training.

</details>